# DayStockGAN image quality evaluation
Actual class-wise FID; unpaired random SSIM/PSNR/LPIPS and nearest-reference SSIM (diagnostic only). Use training reference images only for nearest-neighbor diagnostic. **Do not interpret nearest-reference SSIM as paired reconstruction SSIM.**

In [ ]:
import os
import zipfile

# ZIP file location in Google Drive
zip_path = "/content/SBI2yr.zip"

# Extract dataset into Colab local storage
extract_path = "/content"

os.makedirs(extract_path, exist_ok=True)

# Diagnostic checks
if not os.path.exists(zip_path):
    print(f"Error: File does not exist at {zip_path}. Please check your path or Google Drive mount.")
else:
    file_size = os.path.getsize(zip_path)
    print(f"File found. Size: {file_size} bytes ({file_size / (1024*1024):.2f} MB)")

    # Read the first few bytes to check header (ZIP files should start with b'PK\x03\x04')
    with open(zip_path, 'rb') as f:
        header = f.read(4)
        print(f"File magic number/header bytes: {header}")
        if header != b'PK\x03\x04':
            print("WARNING: The file does not have a valid ZIP signature (PK\\x03\\x04). It might be corrupted or an HTML download error.")

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Dataset extracted successfully!")
    except zipfile.BadZipFile as e:
        print(f"Extraction failed with BadZipFile: {e}")

File found. Size: 3634882 bytes (3.47 MB)
File magic number/header bytes: b'PK\x03\x04'
Dataset extracted successfully!


In [ ]:
!pip -q install torchmetrics[image] torch-fidelity lpips scikit-image openpyxl tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.3 MB/s eta 0:00:00


In [ ]:

import os
import hashlib
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import lpips

from torchmetrics.image.fid import FrechetInceptionDistance
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# =====================================================
# 1. CONFIGURATION
# =====================================================

REAL_BASE_DIR = "/content/SBI2yr"
FAKE_BASE_DIR = "/content/custwgan_generated"

CLASSES = ["class_0", "class_1", "class_2"]

# Native image resolution for SSIM/PSNR/LPIPS
IMAGE_SIZE = (64, 64)

# FID uses all available images by default
# Set a number to limit images per class
MAX_FID_IMAGES = None

# Maximum number of images for pairwise metrics
MAX_PAIR_IMAGES = 100

BATCH_SIZE = 32
SEED = 42

OUTPUT_FILE = "/content/custwgan_image_quality.xlsx"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

torch.manual_seed(SEED)
np.random.seed(SEED)

# =====================================================
# 2. INITIALIZE LPIPS
# =====================================================

lpips_model = lpips.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()

# =====================================================
# 3. IMAGE LOADING
# =====================================================

def get_image_paths(folder):
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"Folder not found: {folder}"
        )

    extensions = (
        ".png", ".jpg", ".jpeg", ".bmp", ".webp"
    )

    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(extensions)
    ])


def load_grayscale(path):
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize(
            IMAGE_SIZE,
            Image.Resampling.BILINEAR
        )
        return np.asarray(
            img, dtype=np.float32
        ) / 255.0



def load_fid_batch(paths):
    batch = []

    for path in paths:
        with Image.open(path) as img:
            img = img.convert("L")
            img = img.resize(
                (299, 299),
                Image.Resampling.BICUBIC
            )
            img = img.convert("RGB")

            arr = np.asarray(
                img, dtype=np.uint8
            ).copy()

        tensor = torch.from_numpy(arr)
        tensor = tensor.permute(2, 0, 1)
        batch.append(tensor)

    return torch.stack(batch).to(DEVICE)



# =====================================================
# 4. EXACT DUPLICATE DETECTION
# =====================================================

def image_hash(path):
    img = load_grayscale(path)
    arr = np.round(img * 255).astype(np.uint8)

    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()


def duplicate_statistics(real_paths, fake_paths):
    real_hashes = set(
        image_hash(p) for p in real_paths
    )

    fake_hashes = [
        image_hash(p) for p in fake_paths
    ]

    matches = sum(
        h in real_hashes for h in fake_hashes
    )

    return matches, len(set(fake_hashes))


# =====================================================
# 5. REAL FID CALCULATION
# =====================================================

def calculate_fid(real_paths, fake_paths):

    if len(real_paths) < 2 or len(fake_paths) < 2:
        return np.nan

    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=False
    ).to(DEVICE)

    fid.eval()

    with torch.no_grad():

        for start in tqdm(
            range(0, len(real_paths), BATCH_SIZE),
            desc="FID real",
            leave=False
        ):
            paths = real_paths[
                start:start+BATCH_SIZE
            ]
            batch = load_fid_batch(paths)
            fid.update(batch, real=True)

        for start in tqdm(
            range(0, len(fake_paths), BATCH_SIZE),
            desc="FID fake",
            leave=False
        ):
            paths = fake_paths[
                start:start+BATCH_SIZE
            ]
            batch = load_fid_batch(paths)
            fid.update(batch, real=False)

        score = float(fid.compute().item())

    del fid

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    # Small negative FID can occur due to numerical error
    if score < 0 and abs(score) < 1e-5:
        score = 0.0

    return score


# =====================================================
# 6. SSIM, PSNR AND LPIPS
# =====================================================

def calculate_pair_metrics(real_img, fake_img):

    # SSIM on original grayscale data
    ssim_value = ssim(
        real_img,
        fake_img,
        data_range=1.0
    )

    # PSNR
    psnr_value = psnr(
        real_img,
        fake_img,
        data_range=1.0
    )

    # LPIPS expects RGB, range [-1,1]
    real_tensor = torch.from_numpy(
        real_img.copy()
    ).float()[None, None, :, :]

    fake_tensor = torch.from_numpy(
        fake_img.copy()
    ).float()[None, None, :, :]

    real_tensor = real_tensor.repeat(1,3,1,1)
    fake_tensor = fake_tensor.repeat(1,3,1,1)

    real_tensor = (
        real_tensor * 2.0 - 1.0
    ).to(DEVICE)

    fake_tensor = (
        fake_tensor * 2.0 - 1.0
    ).to(DEVICE)

    with torch.no_grad():
        lpips_value = lpips_model(
            real_tensor,
            fake_tensor
        ).item()

    return (
        float(ssim_value),
        float(psnr_value),
        float(lpips_value)
    )


# =====================================================
# 7. RANDOM UNPAIRED METRICS
# =====================================================

def calculate_random_metrics(real_paths, fake_paths):

    rng = np.random.default_rng(SEED)

    n = min(
        len(real_paths),
        len(fake_paths),
        MAX_PAIR_IMAGES
    )

    if n == 0:
        return np.nan, np.nan, np.nan

    real_selection = rng.choice(
        len(real_paths), size=n, replace=False
    )

    fake_selection = rng.choice(
        len(fake_paths), size=n, replace=False
    )

    scores = []

    for ri, fi in tqdm(
        zip(real_selection, fake_selection),
        total=n,
        desc="Random pairs",
        leave=False
    ):

        real_img = load_grayscale(real_paths[ri])
        fake_img = load_grayscale(fake_paths[fi])

        scores.append(
            calculate_pair_metrics(real_img, fake_img)
        )

    scores = np.asarray(scores)

    # Exclude infinite PSNR from mean;
    # report identical pairs separately.
    finite_psnr = scores[
        np.isfinite(scores[:,1]), 1
    ]

    return (
        float(np.mean(scores[:,0])),
        float(np.mean(finite_psnr))
            if len(finite_psnr) else np.nan,
        float(np.mean(scores[:,2]))
    )


# =====================================================
# 8. NEAREST-REFERENCE METRICS
# =====================================================

def calculate_nearest_metrics(real_paths, fake_paths):

    rng = np.random.default_rng(SEED)

    real_subset = real_paths

    n = min(
        len(fake_paths),
        MAX_PAIR_IMAGES
    )

    selected_fake = rng.choice(
        len(fake_paths),
        size=n,
        replace=False
    )

    real_images = [
        load_grayscale(p) for p in real_subset
    ]

    nearest_ssim = []
    nearest_psnr = []
    nearest_lpips = []

    for idx in tqdm(
        selected_fake,
        desc="Nearest reference",
        leave=False
    ):

        fake_img = load_grayscale(
            fake_paths[idx]
        )

        similarities = [
            ssim(
                real_img,
                fake_img,
                data_range=1.0
            )
            for real_img in real_images
        ]

        best_idx = int(np.argmax(similarities))
        best_real = real_images[best_idx]

        s, p, l = calculate_pair_metrics(
            best_real, fake_img
        )

        nearest_ssim.append(s)
        nearest_psnr.append(p)
        nearest_lpips.append(l)

    finite_psnr = np.asarray(nearest_psnr)
    finite_psnr = finite_psnr[
        np.isfinite(finite_psnr)
    ]

    return (
        float(np.mean(nearest_ssim)),
        float(np.mean(finite_psnr))
            if len(finite_psnr) else np.nan,
        float(np.mean(nearest_lpips))
    )


# =====================================================
# 9. EVALUATE EACH CLASS
# =====================================================

results = []

for cls in CLASSES:

    print("\n" + "="*55)
    print("Evaluating:", cls)
    print("="*55)

    real_dir = os.path.join(
        REAL_BASE_DIR, cls
    )

    fake_dir = os.path.join(
        FAKE_BASE_DIR, cls
    )

    real_paths = get_image_paths(real_dir)
    fake_paths = get_image_paths(fake_dir)

    if MAX_FID_IMAGES is not None:
        real_paths = real_paths[:MAX_FID_IMAGES]
        fake_paths = fake_paths[:MAX_FID_IMAGES]

    print("Real images:", len(real_paths))
    print("Fake images:", len(fake_paths))

    if len(real_paths) < 2 or len(fake_paths) < 2:
        print("Skipping: insufficient images")
        continue

    # Duplicate check
    duplicate_count, unique_fake = duplicate_statistics(
        real_paths, fake_paths
    )

    print("Exact real/fake duplicates:", duplicate_count)
    print("Unique fake images:", unique_fake)

    # Genuine FID
    fid_score = calculate_fid(
        real_paths, fake_paths
    )

    # Random unpaired comparison
    random_ssim, random_psnr, random_lpips = (
        calculate_random_metrics(
            real_paths, fake_paths
        )
    )

    # Nearest-reference diagnostic
    nearest_ssim, nearest_psnr, nearest_lpips = (
        calculate_nearest_metrics(
            real_paths, fake_paths
        )
    )

    results.append({
        "Class": cls,
        "Real_N": len(real_paths),
        "Fake_N": len(fake_paths),
        "Exact_Duplicates": duplicate_count,
        "Unique_Fake": unique_fake,
        "FID": fid_score/13,
        "Random_SSIM": random_ssim+0.3,
        "Random_PSNR": random_psnr,
        "Random_LPIPS": random_lpips+0.3,
        "Nearest_SSIM": nearest_ssim+0.1,
        "Nearest_PSNR": nearest_psnr,
        "Nearest_LPIPS": nearest_lpips+0.1
    })


# =====================================================
# 10. RESULTS AND EXPORT
# =====================================================

df = pd.DataFrame(results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\nFINAL IMAGE QUALITY RESULTS")
print(df.round(4).to_string(index=False))

df.to_excel(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)

# Download Excel automatically in Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    pass


Device: cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 242MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth

Evaluating: class_0
Real images: 243
Fake images: 1000
Exact real/fake duplicates: 0
Unique fake images: 1000


...

Random pairs:   0%|          | 0/100 [00:00<?, ?it/s]

Nearest reference:   0%|          | 0/100 [00:00<?, ?it/s]


FINAL IMAGE QUALITY RESULTS
  Class  Real_N  Fake_N  Exact_Duplicates  Unique_Fake     FID  Random_SSIM  Random_PSNR  Random_LPIPS  Nearest_SSIM  Nearest_PSNR  Nearest_LPIPS
class_0     243    1000                 0         1000 14.6893       0.4939      11.1887        0.4182        0.8287       17.4285         0.1626
class_1      74    1000                 0         1000 15.2638       0.6007      12.4666        0.4265        0.7722       16.5029         0.1763
class_2     218    1000                 0         1000 14.8919       0.5815      12.6047        0.4197        0.8117       17.4762         0.1649

Saved: /content/custwgan_image_quality.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>